In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Bidirectional-Cross-Confirm-V1 fixed run

Run all once. Four fresh OFF sources independently calibrate ORIGINAL, C1, C2, and the bidirectional candidate; four fresh evaluation sources each produce OFF, legacy SINGLE46, and legacy MULTI44_46. FULL, DELETE90, and SPEED5_4 are fixed: 16 physical source-arms, 48 saved views, and 192 phase encodes. The candidate averages the two held-out C2 directions and is invalid if either direction is invalid. The user performs the GPU run. Process completion is not a method PASS.

In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = 'a155cbcb56eda37fe00b2d9a995a4fef9b317dcb'
DRIVE_ROOT = Path('/content/drive/MyDrive/Video-WM/Bidirectional-Cross-Confirm-V1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE_ROOT / ('bidirectional_cross_confirm_v1_' + stamp)
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(source_commit=SOURCE_SHA, python=sys.version, executable=sys.executable, status='SETUP_STARTED'), indent=2))
print('fixed-run output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2))
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
print('Python:', sys.version, 'Executable:', sys.executable, flush=True)
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'], check=True)
logged_run(['apt-get', 'update', '-qq'], check=True)
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if version('torch') != '2.11.0+cu128':
    logged_run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
logged_run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__) == '2.11.0+cu128', torch.__version__
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
check_code = check_code.replace("assert str(torch.__version__)", "from pathlib import Path\ninfo = dict(python=sys.version, executable=sys.executable, packages={})\nfor package in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):\n    try: info['packages'][package] = importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError: info['packages'][package] = None\nimport json\nPath(RECEIPT_PATH).write_text(json.dumps(info, indent=2))\n".replace('RECEIPT_PATH', repr(str(OUTPUT / 'environment_setup.json'))) + "assert str(torch.__version__)")
logged_run([sys.executable, '-u', '-c', check_code], check=True)


In [ ]:
import subprocess
REPO = Path('/content/SC-SSTW-Bidirectional-Cross-Confirm-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_SHA
print('source commit:', actual, flush=True)


In [ ]:
import subprocess, sys
check_code = """
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('This fixed real Wan run requires a CUDA runtime')
print('device:', torch.cuda.get_device_name(0), flush=True)
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import json, os, subprocess, sys
CONFIG = REPO / 'experiments/wan_state_clock/configs/bidirectional_cross_confirm_v1.json'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.bidirectional_cross_confirm_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
print('fixed experiment argv:', command, flush=True)
print('fixed experiment output:', OUTPUT, flush=True)
completed = subprocess.run(command, cwd=REPO, env=env, check=False)
print('fixed experiment returncode:', completed.returncode, flush=True)
(OUTPUT / 'execution_receipt.json').write_text(json.dumps(dict(command=command, returncode=completed.returncode, result_path=str(OUTPUT / 'result.json')), indent=2))
if not (OUTPUT / 'result.json').exists():
    raise FileNotFoundError('fixed runner produced no retained result.json')


In [ ]:
import json
result_path = OUTPUT / 'result.json'
try:
    result = json.loads(result_path.read_text())
except Exception as exc:
    print('result.json is missing or invalid:', repr(exc), 'path:', result_path, flush=True)
    raise
print('status:', result['status'])
print('fixed denominator:', result.get('fixed_denominator', 'NOT_FINALIZED'))
print('call accounting:', result.get('call_accounting', 'NOT_FINALIZED'))
print('calibrations:', result.get('calibrations', 'NOT_FINALIZED'))
cases = result.get('cases', {})
receiver_names = ('ORIGINAL', 'C1_MATCHED_CONFIRM', 'C2_STATE_CONFIRM', 'BIDIRECTIONAL_C2_MEAN')
view_statuses = {receiver:{} for receiver in receiver_names}; source_statuses = {receiver:{} for receiver in receiver_names}
for case_id, case in cases.items():
    print('case:', case_id, 'status:', case.get('status'), 'generate_exit:', case.get('generate_exit_code'), 'media_exit:', case.get('media_exit_code'))
    print(' case failures:', case.get('failures', []), 'parent failures:', case.get('parent_failures', []))
    for arm in case.get('videos', {}).values():
        for receiver, decision in arm.get('receiver_source_decisions', {}).items():
            status = decision.get('status', 'NOT_RUN'); bucket = source_statuses.setdefault(receiver, {}); bucket[status] = bucket.get(status, 0) + 1
        for view in arm.get('views', {}).values():
            for receiver, row in view.get('receivers', {}).items():
                status = row.get('decision', {}).get('status', 'NOT_RUN'); bucket = view_statuses.setdefault(receiver, {}); bucket[status] = bucket.get(status, 0) + 1
for receiver in receiver_names:
    print('receiver slots:', receiver, dict(view_slots=sum(view_statuses[receiver].values()), expected_view_slots=48, view_statuses=view_statuses[receiver], source_slots=sum(source_statuses[receiver].values()), expected_source_slots=16, source_statuses=source_statuses[receiver]))
evaluation = {receiver:{'OFF_views':{}, 'marked_views':{}, 'OFF_sources':{}, 'marked_sources':{}} for receiver in receiver_names}
for case in cases.values():
    if case.get('role') != 'evaluation': continue
    for arm_name, arm in case.get('videos', {}).items():
        kind = 'OFF' if arm_name == 'OFF' else 'marked'
        for receiver, decision in arm.get('receiver_source_decisions', {}).items():
            bucket = evaluation[receiver][kind + '_sources']; status = decision.get('status', 'NOT_RUN'); bucket[status] = bucket.get(status, 0) + 1
        for view in arm.get('views', {}).values():
            for receiver, row in view.get('receivers', {}).items():
                bucket = evaluation[receiver][kind + '_views']; status = row.get('decision', {}).get('status', 'NOT_RUN'); bucket[status] = bucket.get(status, 0) + 1
for receiver in receiver_names:
    print('evaluation-only:', receiver, evaluation[receiver], 'expected:', dict(OFF_views=12, marked_views=24, OFF_sources=4, marked_sources=8))
for row in result.get('comparison_records', []):
    print('comparison:', row)
print('top-level retained failures:', result.get('failures', []))
print('full result:', result_path)
